# SuperDL – HUF bankjegy YOLO tréning (Colab GPU)

**Használat:**
1. Töltsd fel a `banknote_yolo_dataset.zip` fájlt a Google Drive **gyökerébe** (My Drive).
2. Fent: **Runtime → Change runtime type → T4 GPU**.
3. Futtasd a cellákat sorban (mindegyik bal oldalán a ▶ gomb).

A kész modell a Drive-odra kerül: `MyDrive/SuperDL_model/`

## 1. GPU ellenőrzés

In [ ]:
!nvidia-smi
# Ha ez hibat ad vagy 'not found', akkor nincs GPU:
# Runtime -> Change runtime type -> T4 GPU, majd ujra ezt a cellat.

## 2. Google Drive csatolása

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 3. Ultralytics (YOLO) telepítése

In [ ]:
!pip install -q ultralytics
import ultralytics
ultralytics.checks()

## 4. Adathalmaz kicsomagolása a Drive-rol

In [ ]:
import zipfile, os

ZIP_PATH = '/content/drive/MyDrive/banknote_yolo_dataset.zip'
EXTRACT_TO = '/content/dataset'

assert os.path.exists(ZIP_PATH), f'NEM TALALHATO: {ZIP_PATH} -- toltsd fel a ZIP-et a Drive gyokerebe!'

os.makedirs(EXTRACT_TO, exist_ok=True)
print('Kicsomagolas...')
with zipfile.ZipFile(ZIP_PATH, 'r') as z:
    z.extractall(EXTRACT_TO)
print('Kesz.')
!ls -la /content/dataset/banknote_yolo

## 5. data.yaml útvonalak Colab-ra igazítása
A ZIP-ben lévő data.yaml Windows-útvonalakat tartalmaz, ezeket felülírjuk a Colab útvonalaira.

In [ ]:
yaml_text = '''path: /content/dataset/banknote_yolo
train: images/train
val: images/val
test: images/test
nc: 6
names:
  0: huf_500
  1: huf_1000
  2: huf_2000
  3: huf_5000
  4: huf_10000
  5: huf_20000
'''
with open('/content/dataset/banknote_yolo/data.yaml', 'w') as f:
    f.write(yaml_text)
print('data.yaml frissitve Colab utvonalakra.')

## 6. TRÉNING
150 epoch, 640px, patience=25 (magatol leall ha 25 epochig nem javul).

**Idő:** T4 GPU-n kb. 1-3 óra. A checkpoint minden epoch vegen mentodik, tehat megszakithato.

In [ ]:
from ultralytics import YOLO

model = YOLO('yolo11s.pt')
results = model.train(
    data='/content/dataset/banknote_yolo/data.yaml',
    imgsz=640,
    epochs=150,
    patience=25,
    batch=16,
    project='/content/runs',
    name='huf_detect',
    exist_ok=True
)

## 7. Validálás (pontosság ellenőrzés)

In [ ]:
best = YOLO('/content/runs/huf_detect/weights/best.pt')
metrics = best.val(data='/content/dataset/banknote_yolo/data.yaml')
print('mAP50:', metrics.box.map50)
print('mAP50-95:', metrics.box.map)

## 8. Export TFLite (Android) -- FLOAT32 (az app ezt varja)

In [ ]:
best = YOLO('/content/runs/huf_detect/weights/best.pt')
best.export(format='tflite', imgsz=640, int8=False,
            data='/content/dataset/banknote_yolo/data.yaml')

## 9. Kész modell mentése a Drive-ra
Ez a legfontosabb cella – enelkul a session vegen minden elveszik!

In [ ]:
import shutil, glob, os

OUT = '/content/drive/MyDrive/SuperDL_model'
os.makedirs(OUT, exist_ok=True)

# best.pt (tovabbi tanitashoz / resume-hoz)
shutil.copy('/content/runs/huf_detect/weights/best.pt', f'{OUT}/best.pt')

# TFLite fajl(ok) megkeresese es masolasa
tflite_files = glob.glob('/content/runs/huf_detect/weights/*.tflite')
for f in tflite_files:
    shutil.copy(f, OUT)
    print('Mentve:', os.path.basename(f))

print('\nMINDEN MENTVE IDE:', OUT)
!ls -la {OUT}

---
## Ha később FOLYTATNI akarod (nem elölről)
Ha megszakadt a tréning, töltsd fel a Drive-ról a `best.pt`-t, és a 6. cellában cseréld ki:
```python
model = YOLO('/content/drive/MyDrive/SuperDL_model/best.pt')
results = model.train(..., resume=True)
```